In [16]:

"""IMDB_Sentiment_Analysis.ipynb

A sentiment analysis model trained on the IMDB dataset
"""

'IMDB_Sentiment_Analysis.ipynb\n\nA sentiment analysis model trained on the IMDB dataset\n'

In [17]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
!pip install -q kagglehub

In [19]:
!mkdir -p ~/.kaggle

In [20]:
import kagglehub

In [21]:


# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)
# df = pd.read_csv("/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")
# print(df.shape)

Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [22]:
df = pd.read_csv("/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")
print(df.shape)

(50000, 2)


In [23]:
df.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [24]:
df["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [25]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})


In [26]:
df["sentiment"].value_counts()


,count
sentiment,
1,25000
0,25000


In [27]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(df, test_size=0.20, random_state=42, stratify=df['sentiment'])
val, test = train_test_split(temp, test_size=0.50, random_state=42, stratify=temp['sentiment'])
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 40000, Val: 5000, Test: 5000


In [28]:
train_data = train
val_data = val
test_data = test

In [29]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


# Tokenize text data
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data["review"])

# Convert text to sequences and pad
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200)
X_val = pad_sequences(tokenizer.texts_to_sequences(val_data["review"]), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)

# Target variables
Y_train = train_data["sentiment"].values
Y_val = val_data["sentiment"].values
Y_test = test_data["sentiment"].values

In [ ]:
# This may take 10-15 min to fully run.
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from zipfile import ZipFile

model = Sequential()
model.add(Embedding(input_dim=5001, output_dim=128))
model.add(LSTM(units=64, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(units=1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train, Y_train,
    epochs=9,
    batch_size=128,
    validation_data=(X_val, Y_val),
    callbacks=[early_stopping]
)

model.summary()

Epoch 1/9
 21/313 ━━━━━━━━━━━━━━━━━━━━ 2:34 530ms/step - accuracy: 0.5239 - loss: 0.6920

In [ ]:
model.save("sentiment_imdb_model.keras")
print("Model saved successfully.")

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("sentiment_imdb_model.keras")

loss, accuracy = model.evaluate(X_test, Y_test)
print(f"Model Loss: {loss:.4f}")
print(f"Model Accuracy: {accuracy:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', marker='o')
plt.plot(history.history['val_accuracy'], label='Val Accuracy', linestyle='--', marker='o')  # Dotted line
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', marker='o')
plt.plot(history.history['val_loss'], label='Val Loss', linestyle='--', marker='o')  # Dotted line
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns
import matplotlib.pyplot as plt

y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype("int32")

cm = confusion_matrix(Y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
def predict_sentiment(review):
    # Preprocess the review
    review = review.lower().strip()
    # Tokenize and pad
    sequence = tokenizer.texts_to_sequences([review])
    padded_sequence = pad_sequences(sequence, maxlen=200)
    # Predict sentiment
    prediction = model.predict(padded_sequence)
    prob = prediction[0][0]
    sentiment = "positive" if prob > 0.5 else "negative"
    return sentiment, prob

In [ ]:
new_review = "This movie was amazing, I loved it so much!"
sentiment, confidence = predict_sentiment(new_review)
print(f"The sentiment is: {sentiment} (confidence: {confidence:.2f})")

In [ ]:
new_review = "This movie was average for others, but I really loved it!"
sentiment, confidence = predict_sentiment(new_review)
print(f"The sentiment is: {sentiment} (confidence: {confidence:.2f})")

In [ ]:
new_review = "disgusting"
sentiment, confidence = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment} (confidence: {confidence:.2f})")

In [ ]:
print(f"Model test accuracy: {accuracy:.4f}")